# Spinlock Performance Analysis

This notebook analyzes the performance of different spinlock implementations:
- TAS (Test-And-Set)
- TTAS (Test-Test-And-Set)
- TTAS Optimal (with backoff)
- Ticket Lock
- Ticket Lock Optimal (with backoff)

We'll visualize maximum and average execution times as thread count varies.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ctypes
import os
from pathlib import Path

# Configure matplotlib for better plots
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

## Section 1: Load and Call C Functions

Load the compiled spinlock library and define wrapper functions to call the C `get_times()` function.

In [ ]:
# Locate the compiled spinlock library
spinlock_dir = Path("/home/daniil/MIPT_shiz/course_1-4/concurrency/hw/spinlock")
build_dir = spinlock_dir / "build"

# Try to find the compiled shared library
lib_path = None
if (build_dir / "libthread_spinlock_test.so").exists():
    lib_path = build_dir / "libthread_spinlock_test.so"
elif (build_dir / "CMakeFiles").exists():
    # Search for .so files in build directory
    for so_file in build_dir.glob("**/*.so"):
        lib_path = so_file
        break

if lib_path:
    print(f"Found library: {lib_path}")
    # Load the C library
    lib = ctypes.CDLL(str(lib_path))
else:
    print("Warning: Could not find compiled spinlock library. Will attempt to compile or use alternative path.")
    # Try default build artifact name
    try:
        lib = ctypes.CDLL(str(build_dir / "libthread_spinlock_test.so"))
    except Exception as e:
        print(f"Error loading library: {e}")
        lib = None

In [ ]:
# Define wrapper functions for each spinlock implementation
def call_spinlock_tas(thread_num):
    """Call spinlock TAS get_times"""
    if lib is None:
        return None, None
    
    avg_time = ctypes.c_double()
    max_time = ctypes.c_double()
    
    func = lib.spinlock_tas_get_times
    func.argtypes = [ctypes.POINTER(ctypes.c_double), 
                     ctypes.POINTER(ctypes.c_double), 
                     ctypes.c_int]
    func.restype = None
    
    try:
        func(ctypes.byref(avg_time), ctypes.byref(max_time), thread_num)
        return avg_time.value, max_time.value
    except Exception as e:
        print(f"Error in TAS: {e}")
        return None, None

def call_spinlock_ttas(thread_num):
    """Call spinlock TTAS get_times"""
    if lib is None:
        return None, None
    
    avg_time = ctypes.c_double()
    max_time = ctypes.c_double()
    
    func = lib.spinlock_ttas_get_times
    func.argtypes = [ctypes.POINTER(ctypes.c_double), 
                     ctypes.POINTER(ctypes.c_double), 
                     ctypes.c_int]
    func.restype = None
    
    try:
        func(ctypes.byref(avg_time), ctypes.byref(max_time), thread_num)
        return avg_time.value, max_time.value
    except Exception as e:
        print(f"Error in TTAS: {e}")
        return None, None

def call_spinlock_ttas_optimal(thread_num):
    """Call spinlock TTAS Optimal get_times"""
    if lib is None:
        return None, None
    
    avg_time = ctypes.c_double()
    max_time = ctypes.c_double()
    
    func = lib.spinlock_ttas_optimal_get_times
    func.argtypes = [ctypes.POINTER(ctypes.c_double), 
                     ctypes.POINTER(ctypes.c_double), 
                     ctypes.c_int]
    func.restype = None
    
    try:
        func(ctypes.byref(avg_time), ctypes.byref(max_time), thread_num)
        return avg_time.value, max_time.value
    except Exception as e:
        print(f"Error in TTAS Optimal: {e}")
        return None, None

def call_ticket_lock(thread_num):
    """Call ticket lock get_times"""
    if lib is None:
        return None, None
    
    avg_time = ctypes.c_double()
    max_time = ctypes.c_double()
    
    func = lib.ticket_lock_get_times
    func.argtypes = [ctypes.POINTER(ctypes.c_double), 
                     ctypes.POINTER(ctypes.c_double), 
                     ctypes.c_int]
    func.restype = None
    
    try:
        func(ctypes.byref(avg_time), ctypes.byref(max_time), thread_num)
        return avg_time.value, max_time.value
    except Exception as e:
        print(f"Error in Ticket Lock: {e}")
        return None, None

def call_ticket_lock_optimal(thread_num):
    """Call ticket lock optimal get_times"""
    if lib is None:
        return None, None
    
    avg_time = ctypes.c_double()
    max_time = ctypes.c_double()
    
    func = lib.ticket_lock_optimal_get_times
    func.argtypes = [ctypes.POINTER(ctypes.c_double), 
                     ctypes.POINTER(ctypes.c_double), 
                     ctypes.c_int]
    func.restype = None
    
    try:
        func(ctypes.byref(avg_time), ctypes.byref(max_time), thread_num)
        return avg_time.value, max_time.value
    except Exception as e:
        print(f"Error in Ticket Lock Optimal: {e}")
        return None, None

In [ ]:
# Test with multiple thread counts for all spinlock implementations
thread_counts = [4, 16, 32, 64, 128]
results = []

# Define the spinlock implementations
spinlocks = [
    ('TAS', call_spinlock_tas),
    ('TTAS', call_spinlock_ttas),
    ('TTAS Optimal', call_spinlock_ttas_optimal),
    ('Ticket Lock', call_ticket_lock),
    ('Ticket Lock Optimal', call_ticket_lock_optimal),
]

print("Running performance tests for all spinlock implementations...\n")

for thread_num in thread_counts:
    print(f"{'='*60}")
    print(f"Testing with {thread_num} threads")
    print(f"{'='*60}")
    
    for impl_name, impl_func in spinlocks:
        avg_time, max_time = impl_func(thread_num)
        if avg_time is not None:
            results.append({
                'implementation': impl_name,
                'thread_count': thread_num,
                'avg_time': avg_time,
                'max_time': max_time
            })
            print(f"✓ {impl_name:25s} | Avg: {avg_time:10.4f} us | Max: {max_time:10.4f} us")
        else:
            print(f"✗ {impl_name:25s} | FAILED")
    print()

print(f"\nTest completed!")
print(f"Collected {len(results)} data points")

## Section 2: Organize Performance Data

Convert the collected results into a pandas DataFrame for easier analysis and visualization.

In [ ]:
# Create DataFrame from results
if results:
    df = pd.DataFrame(results)
    print("Performance Data (All Implementations):")
    print("="*80)
    print(df.to_string(index=False))
    print("\n" + "="*80)
    print(f"DataFrame shape: {df.shape}")
    print(f"Implementations: {df['implementation'].unique().tolist()}")
    print(f"Thread counts: {sorted(df['thread_count'].unique().tolist())}")
else:
    print("No results collected. Check if the library was loaded correctly.")

## Section 3: Visualize Maximum Time by Thread Count

Plot how maximum execution time varies with the number of threads.

In [ ]:
if results:
    fig, ax = plt.subplots(figsize=(14, 7))
    
    # Group by implementation
    implementations = df['implementation'].unique()
    colors = ['red', 'blue', 'green', 'orange', 'purple']
    
    for impl, color in zip(implementations, colors):
        impl_data = df[df['implementation'] == impl].sort_values('thread_count')
        ax.plot(impl_data['thread_count'], impl_data['max_time'], 
                marker='o', linewidth=2, markersize=8, label=impl, color=color)
    
    ax.set_xlabel('Number of Threads', fontsize=12, fontweight='bold')
    ax.set_ylabel('Maximum Execution Time (us)', fontsize=12, fontweight='bold')
    ax.set_title('Spinlock Performance: Maximum Time vs Thread Count', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=11, loc='best')
    ax.set_xticks(sorted(df['thread_count'].unique()))
    
    plt.tight_layout()
    plt.savefig('/home/daniil/MIPT_shiz/course_1-4/concurrency/hw/spinlock/analysis/max_time_all_implementations.png', dpi=150)
    plt.show()
    
    print("✓ Maximum time plot saved!")

## Section 4: Visualize Average Time by Thread Count

Plot how average execution time varies with the number of threads.

In [ ]:
if results:
    fig, ax = plt.subplots(figsize=(14, 7))
    
    # Group by implementation
    implementations = df['implementation'].unique()
    colors = ['red', 'blue', 'green', 'orange', 'purple']
    
    for impl, color in zip(implementations, colors):
        impl_data = df[df['implementation'] == impl].sort_values('thread_count')
        ax.plot(impl_data['thread_count'], impl_data['avg_time'], 
                marker='s', linewidth=2, markersize=8, label=impl, color=color)
    
    ax.set_xlabel('Number of Threads', fontsize=12, fontweight='bold')
    ax.set_ylabel('Average Execution Time (us)', fontsize=12, fontweight='bold')
    ax.set_title('Spinlock Performance: Average Time vs Thread Count', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=11, loc='best')
    ax.set_xticks(sorted(df['thread_count'].unique()))
    
    plt.tight_layout()
    plt.savefig('/home/daniil/MIPT_shiz/course_1-4/concurrency/hw/spinlock/analysis/avg_time_all_implementations.png', dpi=150)
    plt.show()
    
    print("✓ Average time plot saved!")

## Section 5: Compare Both Metrics

Create a combined plot showing both average and maximum times on the same chart for easier comparison.

In [ ]:
if results:
    implementations = sorted(df['implementation'].unique())
    n_impl = len(implementations)
    
    # Create subplots for each implementation
    fig, axes = plt.subplots(n_impl, 2, figsize=(16, 4*n_impl))
    
    colors = ['red', 'blue', 'green', 'orange', 'purple']
    
    for idx, (impl, color) in enumerate(zip(implementations, colors)):
        impl_data = df[df['implementation'] == impl].sort_values('thread_count')
        
        # Plot max_time
        axes[idx, 0].plot(impl_data['thread_count'], impl_data['max_time'], 
                         marker='o', linewidth=2, markersize=8, color=color)
        axes[idx, 0].set_ylabel('Max Time (us)', fontsize=11)
        axes[idx, 0].set_title(f'{impl} - Maximum Time', fontsize=12, fontweight='bold')
        axes[idx, 0].grid(True, alpha=0.3)
        axes[idx, 0].set_xticks(sorted(df['thread_count'].unique()))
        
        # Plot avg_time
        axes[idx, 1].plot(impl_data['thread_count'], impl_data['avg_time'], 
                         marker='s', linewidth=2, markersize=8, color=color)
        axes[idx, 1].set_ylabel('Avg Time (us)', fontsize=11)
        axes[idx, 1].set_title(f'{impl} - Average Time', fontsize=12, fontweight='bold')
        axes[idx, 1].grid(True, alpha=0.3)
        axes[idx, 1].set_xticks(sorted(df['thread_count'].unique()))
    
    # Set x-label only for bottom row
    for ax in axes[-1]:
        ax.set_xlabel('Number of Threads', fontsize=11, fontweight='bold')
    
    plt.suptitle('Spinlock Performance Comparison - All Implementations', 
                 fontsize=14, fontweight='bold', y=1.001)
    plt.tight_layout()
    plt.savefig('/home/daniil/MIPT_shiz/course_1-4/concurrency/hw/spinlock/analysis/all_implementations_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("✓ Comparison plot saved!")

## Section 6: Summary Statistics

Display summary statistics to help understand the performance characteristics.

In [ ]:
if results:
    print("\n" + "="*80)
    print("SUMMARY STATISTICS FOR ALL IMPLEMENTATIONS")
    print("="*80 + "\n")
    
    for impl in sorted(df['implementation'].unique()):
        impl_data = df[df['implementation'] == impl]
        print(f"\n{impl.upper()}")
        print("-" * 60)
        print(impl_data[['thread_count', 'avg_time', 'max_time']].to_string(index=False))
        print(f"\n  Avg (across all threads):")
        print(f"    Average time: {impl_data['avg_time'].mean():.4f} us")
        print(f"    Max time:     {impl_data['max_time'].mean():.4f} us")
        print(f"  Max values across all threads:")
        print(f"    Peak avg_time: {impl_data['avg_time'].max():.4f} us")
        print(f"    Peak max_time: {impl_data['max_time'].max():.4f} us")
    
    print("\n" + "="*80)
    print("CROSS-IMPLEMENTATION COMPARISON")
    print("="*80 + "\n")
    
    summary_stats = df.groupby('implementation').agg({
        'avg_time': ['mean', 'min', 'max'],
        'max_time': ['mean', 'min', 'max']
    }).round(4)
    print(summary_stats)
    
    print("\n" + "="*80)
    print("BEST PERFORMER BY THREAD COUNT")
    print("="*80 + "\n")
    
    for thread_count in sorted(df['thread_count'].unique()):
        thread_data = df[df['thread_count'] == thread_count]
        best_avg = thread_data.loc[thread_data['avg_time'].idxmin()]
        best_max = thread_data.loc[thread_data['max_time'].idxmin()]
        print(f"Threads: {thread_count:2d}")
        print(f"  Lowest avg_time:  {best_avg['implementation']:25s} ({best_avg['avg_time']:.4f} us)")
        print(f"  Lowest max_time:  {best_max['implementation']:25s} ({best_max['max_time']:.4f} us)")
    
    print("\n" + "="*80)

## Section 7: Detailed Comparison - TTAS Optimal vs Ticket Lock Optimal

Direct comparison of the two best-performing spinlock implementations.

In [ ]:
if results:
    # Filter data for only the two best implementations
    best_impls = ['TTAS Optimal', 'Ticket Lock Optimal']
    df_best = df[df['implementation'].isin(best_impls)]
    
    fig, ax = plt.subplots(figsize=(12, 6))
    
    colors = {'TTAS Optimal': 'green', 'Ticket Lock Optimal': 'purple'}
    
    for impl in best_impls:
        impl_data = df_best[df_best['implementation'] == impl].sort_values('thread_count')
        ax.plot(impl_data['thread_count'], impl_data['max_time'], 
                marker='o', linewidth=2.5, markersize=10, label=impl, color=colors[impl])
    
    ax.set_xlabel('Number of Threads', fontsize=12, fontweight='bold')
    ax.set_ylabel('Maximum Execution Time (us)', fontsize=12, fontweight='bold')
    ax.set_title('Best Performers: Maximum Time Comparison\n(TTAS Optimal vs Ticket Lock Optimal)', 
                fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=12, loc='best')
    ax.set_xticks(sorted(df['thread_count'].unique()))
    
    plt.tight_layout()
    plt.savefig('/home/daniil/MIPT_shiz/course_1-4/concurrency/hw/spinlock/analysis/best_two_max_time.png', dpi=150)
    plt.show()
    
    print("✓ Best two implementations - Maximum time plot saved!")

In [ ]:
if results:
    # Filter data for only the two best implementations
    best_impls = ['TTAS Optimal', 'Ticket Lock Optimal']
    df_best = df[df['implementation'].isin(best_impls)]
    
    fig, ax = plt.subplots(figsize=(12, 6))
    
    colors = {'TTAS Optimal': 'green', 'Ticket Lock Optimal': 'purple'}
    
    for impl in best_impls:
        impl_data = df_best[df_best['implementation'] == impl].sort_values('thread_count')
        ax.plot(impl_data['thread_count'], impl_data['avg_time'], 
                marker='s', linewidth=2.5, markersize=10, label=impl, color=colors[impl])
    
    ax.set_xlabel('Number of Threads', fontsize=12, fontweight='bold')
    ax.set_ylabel('Average Execution Time (us)', fontsize=12, fontweight='bold')
    ax.set_title('Best Performers: Average Time Comparison\n(TTAS Optimal vs Ticket Lock Optimal)', 
                fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=12, loc='best')
    ax.set_xticks(sorted(df['thread_count'].unique()))
    
    plt.tight_layout()
    plt.savefig('/home/daniil/MIPT_shiz/course_1-4/concurrency/hw/spinlock/analysis/best_two_avg_time.png', dpi=150)
    plt.show()
    
    print("✓ Best two implementations - Average time plot saved!")

In [ ]:
if results:
    # Filter data for only the two best implementations
    best_impls = ['TTAS Optimal', 'Ticket Lock Optimal']
    df_best = df[df['implementation'].isin(best_impls)]
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    colors = {'TTAS Optimal': 'green', 'Ticket Lock Optimal': 'purple'}
    
    for impl in best_impls:
        impl_data = df_best[df_best['implementation'] == impl].sort_values('thread_count')
        
        # Max time on left subplot
        axes[0].plot(impl_data['thread_count'], impl_data['max_time'], 
                    marker='o', linewidth=2.5, markersize=10, label=impl, color=colors[impl])
        
        # Avg time on right subplot
        axes[1].plot(impl_data['thread_count'], impl_data['avg_time'], 
                    marker='s', linewidth=2.5, markersize=10, label=impl, color=colors[impl])
    
    # Configure left subplot (Max Time)
    axes[0].set_xlabel('Number of Threads', fontsize=12, fontweight='bold')
    axes[0].set_ylabel('Maximum Execution Time (us)', fontsize=12, fontweight='bold')
    axes[0].set_title('Maximum Time Comparison', fontsize=13, fontweight='bold')
    axes[0].grid(True, alpha=0.3)
    axes[0].legend(fontsize=11, loc='best')
    axes[0].set_xticks(sorted(df['thread_count'].unique()))
    
    # Configure right subplot (Avg Time)
    axes[1].set_xlabel('Number of Threads', fontsize=12, fontweight='bold')
    axes[1].set_ylabel('Average Execution Time (us)', fontsize=12, fontweight='bold')
    axes[1].set_title('Average Time Comparison', fontsize=13, fontweight='bold')
    axes[1].grid(True, alpha=0.3)
    axes[1].legend(fontsize=11, loc='best')
    axes[1].set_xticks(sorted(df['thread_count'].unique()))
    
    plt.suptitle('TTAS Optimal vs Ticket Lock Optimal - Side by Side Comparison', 
                fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('/home/daniil/MIPT_shiz/course_1-4/concurrency/hw/spinlock/analysis/best_two_side_by_side.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("✓ Best two implementations - Side by side comparison plot saved!")

In [ ]:
if results:
    # Filter data for only the two best implementations
    best_impls = ['TTAS Optimal', 'Ticket Lock Optimal']
    df_best = df[df['implementation'].isin(best_impls)]
    
    print("\n" + "="*80)
    print("DETAILED STATISTICS: TTAS OPTIMAL vs TICKET LOCK OPTIMAL")
    print("="*80 + "\n")
    
    for impl in best_impls:
        impl_data = df_best[df_best['implementation'] == impl]
        print(f"\n{impl.upper()}")
        print("-" * 60)
        print(impl_data[['thread_count', 'avg_time', 'max_time']].to_string(index=False))
        print(f"\n  Statistics:")
        print(f"    Avg (avg_time):    {impl_data['avg_time'].mean():.4f} us (μ)")
        print(f"    Avg (max_time):    {impl_data['max_time'].mean():.4f} us (μ)")
        print(f"    Min (avg_time):    {impl_data['avg_time'].min():.4f} us")
        print(f"    Max (avg_time):    {impl_data['avg_time'].max():.4f} us")
        print(f"    Min (max_time):    {impl_data['max_time'].min():.4f} us")
        print(f"    Max (max_time):    {impl_data['max_time'].max():.4f} us")
    
    print("\n" + "="*80)
    print("HEAD-TO-HEAD COMPARISON BY THREAD COUNT")
    print("="*80 + "\n")
    
    print(f"{'Threads':>8} | {'Metric':>12} | {'TTAS Optimal':>15} | {'Ticket Lock Opt':>15} | {'Winner':>15}")
    print("-" * 80)
    
    for thread_count in sorted(df_best['thread_count'].unique()):
        ttas_data = df_best[(df_best['implementation'] == 'TTAS Optimal') & 
                           (df_best['thread_count'] == thread_count)]
        ticket_data = df_best[(df_best['implementation'] == 'Ticket Lock Optimal') & 
                             (df_best['thread_count'] == thread_count)]
        
        if len(ttas_data) > 0 and len(ticket_data) > 0:
            ttas_avg = ttas_data['avg_time'].values[0]
            ticket_avg = ticket_data['avg_time'].values[0]
            ttas_max = ttas_data['max_time'].values[0]
            ticket_max = ticket_data['max_time'].values[0]
            
            winner_avg = "TTAS" if ttas_avg < ticket_avg else "Ticket"
            winner_max = "TTAS" if ttas_max < ticket_max else "Ticket"
            
            print(f"{thread_count:8d} | {'avg_time':>12} | {ttas_avg:15.4f} | {ticket_avg:15.4f} | {winner_avg:>15}")
            print(f"{' ':8s} | {'max_time':>12} | {ttas_max:15.4f} | {ticket_max:15.4f} | {winner_max:>15}")
            print("-" * 80)
    
    print("\n" + "="*80)
    print("PERFORMANCE RATIO (Max/Avg) - Consistency Indicator")
    print("="*80 + "\n")
    
    for impl in best_impls:
        impl_data = df_best[df_best['implementation'] == impl]
        impl_data['ratio'] = impl_data['max_time'] / impl_data['avg_time']
        
        print(f"\n{impl.upper()}")
        print("-" * 60)
        print(impl_data[['thread_count', 'avg_time', 'max_time', 'ratio']].to_string(index=False))
        print(f"\nAverage Ratio: {impl_data['ratio'].mean():.2f}x")
        print(f"Min Ratio:     {impl_data['ratio'].min():.2f}x (most consistent at {impl_data.loc[impl_data['ratio'].idxmin(), 'thread_count']:.0f} threads)")
        print(f"Max Ratio:     {impl_data['ratio'].max():.2f}x (least consistent at {impl_data.loc[impl_data['ratio'].idxmax(), 'thread_count']:.0f} threads)")
    
    print("\n" + "="*80)